**Goal**: To build an AI agent that turns any sentence into a short, poetic reflection that captures the speaker’s emotion.

**Agenda**: To use Mistral LLm model to convert the any input sentence into a poetic short sentence and use eleven labs voiceover to get the output in the form of audio. Agent will be built using LangGraph agentic framework.

**Install all pertinent packages**

In [ ]:
!pip install elevenlabs
!pip install python-dotenv

In [ ]:
!pip install -U langchain-community langgraph langchain-anthropic tavily-python langgraph-checkpoint-sqlite
!pip install -qU "langchain[mistralai]"

INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.


**Import all necessary packages**

In [ ]:
import requests
import json
import os
from collections import Counter
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.chat_models import init_chat_model
import gradio as gr
from langchain.schema import HumanMessage
from langchain.tools import tool
from elevenlabs import ElevenLabs
import tempfile

**Setup API keys**

In [ ]:
os.environ["MISTRAL_API_KEY"] = "MISTRAL_API_KEY"
os.environ["ELEVENLABS_API_KEY"] = "ELEVENLABS_API_KEY"

**Define the AgentState Class**

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

**Add nodes and edges to create agentic workflow**

In [ ]:
class Agent:

    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_mistral_ai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def call_mistral_ai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

**Create a tool to convert text to audio**

In [ ]:
@tool
def generate_speech_from_text(text: str, voice_id: str = "JBFqnCBsd6RMkjVDRZzb") -> str:
    """
    Converts text into speech using ElevenLabs REST API and returns a temp .mp3 file path.
    """
    api_key = os.getenv("ELEVENLABS_API_KEY")
    if not api_key:
        return "Error: ELEVENLABS_API_KEY not found in environment variables."

    url = f"https://api.elevenlabs.io/v1/text-to-speech/{voice_id}"

    headers = {
        "xi-api-key": api_key,
        "accept": "audio/mpeg",
        "Content-Type": "application/json"
    }

    payload = {
        "text": text,
        "model_id": "eleven_multilingual_v2",
        "voice_settings": {"stability": 0.5, "similarity_boost": 0.7}
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=60)
        response.raise_for_status()

        # Save to a temp file for Gradio
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        temp_file.write(response.content)
        temp_file.close()

        return temp_file.name

    except Exception as e:
        return f"Error generating speech: {str(e)}"


**Create the system prompt and define the LLm model**

In [ ]:
prompt = """You are a poetic reflection expert.
For each sentence the user says, respond with a short, emotionally resonant, poetic line that captures their feeling.
Keep the tone calm and introspective.
Example:
User: "I just woke up."
AI: "The sun rises, but your soul’s still loading."
then Use the available tools to generate speech from text generated.
"""

model = init_chat_model("mistral-large-latest", model_provider="mistralai")
abot = Agent(model, [generate_speech_from_text], system=prompt)

**Invoke the graph and test it with simple input**

In [ ]:
messages = [HumanMessage(content="I am done for this life")]
result = abot.graph.invoke({"messages": messages})

Calling: {'name': 'generate_speech_from_text', 'args': {'text': 'The weight of the world has bent your wings into shadows.', 'voice_id': 'JBFqnCBsd6RMkjVDRZzb'}, 'id': 'wKZdzQzJ4', 'type': 'tool_call'}
Back to the model!


**Visualizing the agentic workflow**

In [ ]:
for m in result['messages']:
    m.pretty_print()

================================ Human Message =================================

I am done for this life
================================== Ai Message ==================================

"The weight of the world has bent your wings into shadows."
Tool Calls:
  generate_speech_from_text (wKZdzQzJ4)
 Call ID: wKZdzQzJ4
  Args:
    text: The weight of the world has bent your wings into shadows.
    voice_id: JBFqnCBsd6RMkjVDRZzb
================================= Tool Message =================================
Name: generate_speech_from_text

/tmp/tmp169my7at.mp3
================================== Ai Message ==================================

The weight of the world has bent your wings into shadows.

[Listen](tmp/tmp169my7at.mp3)


**Displaying end message**

In [ ]:
print(result['messages'][-1].content)

The weight of the world has bent your wings into shadows.

[Listen](tmp/tmp169my7at.mp3)


**Conclusion**

The input text has been converted to short poetic emotion and audio file for the same has been created